In [ ]:
# create daily regression fit for predictor features - insitu observation

# filter for observations on class 1 - rain >= 0.1 mm

# WORKFLOW
# 1. Load dfs and format for fitting
# 2. Perform cluster*season wise ML fit
# 3. Check if cluster stratification improves the model
# 4. Check if season stratification improves the model 
# 5. Saved trained regressor model using joblib

In [ ]:
# Libraries
import os
import numpy as np  
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from pathlib import Path

import operator
import scores.categorical

# Paths
train_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_train_df.parquet"
test_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_test_df.parquet"
regressor_model_path = r"..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers"
regressor_metrics_path = r"..\Data\Processed\Bias_correction_daily\Regressor_model\validation_results"
rscv_dir = r"..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results"

os.makedirs(regressor_model_path, exist_ok = True)
os.makedirs(regressor_metrics_path, exist_ok = True)
os.makedirs(rscv_dir, exist_ok = True)

# Configurations
random_state_num = 64
n_cores = 2
target_variable = 'insitu_prec_class'
predictor_features = ['y', 'x', 
        'era5l', 'chirps', 'har', 
        'persiann_prec', 'mswep_prec', 
        'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
        'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed',
        'insitu_kriged', 'cluster', 'season', 'season_num',
        'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 
        'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir',
        'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin', 'doy_cos']

In [ ]:
SEASONS = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11],
}

def get_season(month):
    for s, months in SEASONS.items():
        if month in months:
            return s
    return None

SEASONS_NUM = {
    1: 'winter',
    2: 'pre_monsoon',
    3: 'monsoon',
    4: 'post_monsoon'
}

def get_season_num(season_str):
    for s_n, s_name in SEASONS_NUM.items():
        if s_name == season_str:
            return s_n
    return None

# Helper functions
def get_param_grids():
    param_dist_rf = {
        'n_estimators':      [100, 500, 1000, 2000, 3000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [100, 500, 1000, 2000, 3000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def compute_metrics_regression(y_true, y_pred, label):
    """Compute RMSE, MAE, R² and print with label."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"    [{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  n={len(y_true)}")
    return {'rmse': round(rmse, 4), 'mae': round(mae, 4), 'r2': round(r2, 4),
            'n': len(y_true)}

In [ ]:
# 1. Load and format dfs
train_df = pd.read_parquet(train_df_path)
test_df = pd.read_parquet(test_df_path)

seasons_train_da = train_df['month'].apply(get_season)
train_df['season'] = seasons_train_da
seasons_test_da = test_df['month'].apply(get_season)
test_df['season'] = seasons_test_da

train_df = train_df[train_df['insitu_prec'] >= 0.1]
test_df = test_df[test_df['insitu_prec'] >= 0.1]

cluster_list = sorted(test_df['cluster'].unique().tolist())
season_list = list(SEASONS)

In [ ]:
# 2. First we perform a full cluster*season stratified fit

for cluster in cluster_list:
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        print(f"Processing Cluster {cluster} & Season {season} -----------------------------")

        season_cluster_train = cluster_train[cluster_train['season'] == season].copy()
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()
        
        X_train = season_cluster_train[predictor_features].drop(columns= ['cluster', 'season', 'season_num'], errors='ignore').values
        y_train = season_cluster_train[target_variable].values

        param_dist_rf, param_dist_xgb = get_param_grids()


        for model_type in ['rf', 'xgb']:
            if model_type == 'rf':
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('regressor', RandomForestRegressor(random_state = random_state_num,
                        n_jobs = n_cores))
                ])
                param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
            else:
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('regressor',     XGBRegressor(random_state=random_state_num,
                                    n_jobs=n_cores,
                                    verbosity = 0)),    
                ])
                param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

            search_cv = RandomizedSearchCV(
                estimator = base_model,
                param_distributions = param_dist,
                n_iter = 50,
                cv = 5,
                verbose = 1,
                n_jobs = n_cores,
                scoring = 'neg_root_mean_squared_error'
            )
            search_cv.fit(X_train, y_train)
            best_model = search_cv.best_estimator_
            rscv_results_df = pd.DataFrame(search_cv.cv_results_)
            rscv_path = os.path.join(rscv_dir, f"rscv_result_{cluster}_{season}_{model_type}.csv")
            rscv_results_df.to_csv(rscv_path)

            print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
            print(f"    Best params : {search_cv.best_params_}")
            print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
            print(f"RSCV results saved to : {rscv_path}")
            
            model_path = os.path.join(regressor_model_path, f"{cluster}_{season}_{model_type}.pkl")
            joblib.dump(best_model, model_path)
            print(f"Model  {model_type} saved to {model_path}")

In [ ]:
# 3. Cluster wise only fit

for cluster in cluster_list:
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()
        
    X_train = cluster_train[predictor_features].drop(columns= ['cluster'], errors='ignore').values
    y_train = cluster_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor', RandomForestRegressor(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor',     XGBRegressor(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 50,
            cv = 5,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'neg_root_mean_squared_error'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path = os.path.join(rscv_dir, f"rscv_result_{cluster}_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
            
        model_path = os.path.join(regressor_model_path, f"{cluster}_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")

In [ ]:
# 3. Season wise only fit
for season in season_list:
    season_train = train_df[train_df['season'] == season].copy()
    season_test = test_df[test_df['season'] == season].copy()
        
    X_train = season_train[predictor_features].drop(columns= ['season'], errors='ignore').values
    y_train = season_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor', RandomForestRegressor(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'Regressor__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor',     XGBRegressor(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 50,
            cv = 5,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'neg_root_mean_squared_error'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path  = os.path.join(rscv_dir, f"rscv_result_{season}_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
            
        model_path = os.path.join(regressor_model_path, f"{season}_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")

In [ ]:
# 4. Aggregated fit

train = train_df.copy()
test = test_df.copy()
        
X_train = train[predictor_features].values
y_train = train[target_variable].values

param_dist_rf, param_dist_xgb = get_param_grids()

for model_type in ['rf', 'xgb']:
    if model_type == 'rf':
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('regressor', RandomForestRegressor(random_state = random_state_num,
                n_jobs = n_cores))
        ])
        param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
    else:
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('regressor',     XGBRegressor(random_state=random_state_num,
                            n_jobs=n_cores,
                            verbosity = 0)),    
        ])
        param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

    search_cv = RandomizedSearchCV(
        estimator = base_model,
        param_distributions = param_dist,
        n_iter = 50,
        cv = 5,
        verbose = 1,
        n_jobs = n_cores,
        scoring = 'neg_root_mean_squared_error'
    )
    search_cv.fit(X_train, y_train)
    best_model = search_cv.best_estimator_
    rscv_results_df = pd.DataFrame(search_cv.cv_results_)
    rscv_path = os.path.join(rscv_dir, f"rscv_result_aggregated_{model_type}.csv")
    rscv_results_df.to_csv(rscv_path)

    print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
    print(f"    Best params : {search_cv.best_params_}")
    print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
    print(f"RSCV results saved to : {rscv_path}")
            
    model_path = os.path.join(classifier_model_path, f"Aggregated_{model_type}.pkl")
    joblib.dump(best_model, model_path)
    print(f"Model  {model_type} saved to {model_path}")

In [ ]:
# 5. Validation: per cluster*season subset, compare all four model strategies

# Feature subsets matching how each strategy was trained
feature_sets = {
    'cluster_season': [f for f in predictor_features if f not in ('cluster', 'season')],
    'cluster_only':   [f for f in predictor_features if f != 'cluster'],
    'season_only':    [f for f in predictor_features if f != 'season'],
    'aggregated':     predictor_features,
}

# Model path templates per strategy
def get_model_path(strategy, cluster, season, model_type):
    templates = {
        'cluster_season': f"{cluster}_{season}_{model_type}.pkl",
        'cluster_only':   f"{cluster}_{model_type}.pkl",
        'season_only':    f"{season}_{model_type}.pkl",   
        'aggregated':     f"Aggregated_{model_type}.pkl", 
    }                                                     
    return os.path.join(regressor_model_path, templates[strategy])

all_metrics = []

for cluster in cluster_list:
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()

        if season_cluster_test.empty:
            print(f"[SKIP] No test data — Cluster {cluster} | Season {season}")
            continue

        y_test = season_cluster_test[target_variable].values

        for model_type in ['rf', 'xgb']:
            for strategy, features in feature_sets.items():
                model_path = get_model_path(strategy, cluster, season, model_type)

                if not os.path.exists(model_path):
                    print(f"  [MISSING] {model_path}")
                    continue

                model    = joblib.load(model_path)
                X_test   = season_cluster_test[features].values
                y_pred   = model.predict(X_test)

                label    = f"C{cluster} | {season:<13} | {model_type.upper()} | {strategy}"
                metrics  = compute_metrics_regression(y_test, y_pred, label)
                metrics.update({
                    'cluster':    cluster,
                    'season':     season,
                    'model_type': model_type,
                    'strategy':   strategy,
                })
                all_metrics.append(metrics)

# %%
# Collate and save
metrics_df = pd.DataFrame(all_metrics)

# Pivot for easy visual comparison: rows = cluster*season, cols = strategy*model_type
pivot = metrics_df.pivot_table(
    index   = ['cluster', 'season'],
    columns = ['model_type', 'strategy'],
    values  = ['accuracy', 'f1-score'],
)
pivot.columns = ['_'.join(c) for c in pivot.columns]
pivot = pivot.reset_index()

metrics_out_path = os.path.join(regressor_metrics_path, "validation_comparison.csv")
pivot_out_path   = os.path.join(regressor_metrics_path, "validation_pivot.csv")

metrics_df.to_csv(metrics_out_path, index=False)
pivot.to_csv(pivot_out_path,        index=False)

print(f"\nAll metrics  → {metrics_out_path}")
print(f"Pivot table  → {pivot_out_path}\n")
print(pivot.to_string(index=False))